# JupyterLite で学ぶ DuckDB / SQL 入門チュートリアル

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** 上で、データベース言語 **SQL** の基本を
**DuckDB** を使って学ぶためのチュートリアルです。DuckDB は pandas の DataFrame にそのまま SQL を投げられるので、
「データベースを用意する」手間なしに SQL の練習ができます。

## 対象者
- pandas の基本（DataFrame の作成・表示）を知っている方
- SQL を初めて学ぶ方、または基礎を復習したい方
- 経済・経営データ（売上、店舗、時系列）を SQL で集計してみたい方

## このチュートリアルで学ぶこと
0. 環境準備（JupyterLite 用）
1. SQL と DuckDB とは
2. 練習用データの準備
3. SELECT の基本（列の選択、WHERE、ORDER BY、LIMIT）
4. 集計（GROUP BY、COUNT / SUM / AVG、HAVING）
5. 表の結合（JOIN）
6. サブクエリと CTE（WITH 句）
7. ウィンドウ関数（累積和・移動平均・ランキング）
8. CSV ファイルに直接クエリする
9. インメモリ・データベース（CREATE / INSERT / UPDATE / DELETE）
10. pandas との往復とグラフ化
11. 標準ライブラリ sqlite3 との比較
12. SQL と pandas の対応表
13. まとめと総合演習

## 使い方
- セルを上から順に `Shift + Enter` で実行してください。途中を飛ばすと、表（DataFrame）が未定義でエラーになります。
- 各章の最後に **練習問題** があります。「解答欄」のセルに SQL を書いてから、「解答例」を開いて確認しましょう。
- SQL は大文字・小文字を区別しません。このノートでは、キーワード（SELECT など）を大文字、列名を小文字で書いています。

---
## 0. 環境準備（JupyterLite 用）

DuckDB、pandas、NumPy は JupyterLite（Pyodide）に同梱されています。グラフの日本語表示のために
`japanize-matplotlib-jlite` もインストールします。

In [ ]:
# JupyterLite 用のパッケージインストール（ローカルの Jupyter ではスキップされます）
try:
    import piplite
    await piplite.install(["duckdb", "pandas", "numpy", "matplotlib", "japanize-matplotlib-jlite"])
except ImportError:
    pass

In [ ]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite  # 日本語フォント（plt の後に import する）

print("DuckDB バージョン:", duckdb.__version__)
print("pandas バージョン:", pd.__version__)

---
## 1. SQL と DuckDB とは

**SQL（Structured Query Language）** は、表形式のデータを「選ぶ・絞り込む・集計する・結合する」ための言語です。
企業の販売データや政府統計など、大量のデータは **データベース** に保存され、SQL で取り出すのが一般的です。

**DuckDB** は、分析用途に特化した軽量なデータベースエンジンです。

| 特徴 | 説明 |
|---|---|
| サーバー不要 | `import duckdb` だけで使える（JupyterLite にも同梱） |
| DataFrame に直接 SQL | `duckdb.sql("SELECT ... FROM df")` と書くと、変数 `df` の DataFrame を表として扱える |
| 結果は DataFrame に戻せる | `.df()` で pandas に変換できるので、グラフ化や更なる加工が簡単 |
| CSV / Parquet を直接読める | ファイル名を `FROM 'sales.csv'` のように書くだけ |
| 標準的な SQL | 他のデータベース（PostgreSQL、MySQL、SQLite など）でほぼそのまま使える文法 |

まずは、小さな DataFrame に SQL を投げてみましょう。

In [ ]:
df = pd.DataFrame({
    "name": ["田中", "鈴木", "佐藤", "高橋"],
    "age": [23, 31, 45, 28],
    "city": ["名古屋", "東京", "名古屋", "大阪"],
})

# duckdb.sql() の中では、Python の変数 df をそのまま表として使える
result = duckdb.sql("SELECT name, age FROM df WHERE city = '名古屋'")
print(type(result))
result.df()          # .df() で pandas の DataFrame に変換して表示

`duckdb.sql()` の戻り値は「リレーション（クエリの結果を表すオブジェクト）」で、`.df()` を付けると DataFrame になります。
`.show()` を使うと、SQL らしい罫線付きの表がテキストで表示されます。

In [ ]:
duckdb.sql("SELECT * FROM df ORDER BY age").show()

---
## 2. 練習用データの準備

架空のスーパーの販売データを 3 つの表（テーブル）として作ります。実際のデータベースでも、このように
「商品」「店舗」「売上」を別々の表に分け、ID で結び付けるのが基本です（**正規化**）。

| 表 | 内容 | 主キー |
|---|---|---|
| `products` | 商品マスタ（商品名・カテゴリ・単価） | `product_id` |
| `stores` | 店舗マスタ（店舗名・地域） | `store_id` |
| `sales` | 売上明細（日付・店舗・商品・数量） | `sale_id` |

In [ ]:
rng = np.random.default_rng(42)   # 乱数の種を固定（毎回同じデータになる）

products = pd.DataFrame({
    "product_id": [1, 2, 3, 4, 5, 6, 7],
    "product_name": ["りんご", "バナナ", "みかん", "牛乳", "パン", "コーヒー", "紅茶"],
    "category": ["果物", "果物", "果物", "飲料", "パン", "飲料", "飲料"],
    "price": [128, 98, 60, 210, 180, 350, 300],
})

stores = pd.DataFrame({
    "store_id": [1, 2, 3, 4],
    "store_name": ["名古屋駅前店", "栄店", "豊橋店", "岐阜店"],
    "region": ["愛知", "愛知", "愛知", "岐阜"],
})

n = 300
sales = pd.DataFrame({
    "sale_date": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "store_id": rng.integers(1, 5, n),        # 1〜4
    "product_id": rng.integers(1, 7, n),      # 1〜6（7 の紅茶は一度も売れていない）
    "quantity": rng.integers(1, 10, n),       # 1〜9 個
})
sales = sales.sort_values("sale_date").reset_index(drop=True)
sales.insert(0, "sale_id", np.arange(1, n + 1))

print("products:", products.shape, " stores:", stores.shape, " sales:", sales.shape)

In [ ]:
print(products)
print()
print(stores)

In [ ]:
sales.head(10)

---
## 3. SELECT の基本

SQL の基本形は次のとおりです。各句（clause）は **この順番** で書きます（省略可）。

```sql
SELECT   列1, 列2, ...        -- 取り出す列
FROM     表                   -- どの表から
WHERE    条件                 -- 行の絞り込み
ORDER BY 列 [ASC | DESC]      -- 並べ替え（ASC 昇順 / DESC 降順）
LIMIT    件数                 -- 先頭から何件
```

### 3.1 列を選ぶ

In [ ]:
# * はすべての列
duckdb.sql("SELECT * FROM products").df()

In [ ]:
# 列名を指定して取り出す。AS で別名（エイリアス）を付けられる
duckdb.sql("""
SELECT product_name AS 商品名, price AS 単価
FROM products
""").df()

In [ ]:
# 計算した列も作れる（税込価格）
duckdb.sql("""
SELECT product_name, price, ROUND(price * 1.1) AS price_with_tax
FROM products
""").df()

In [ ]:
# DISTINCT：重複を除いた値の一覧
duckdb.sql("SELECT DISTINCT category FROM products").df()

### 3.2 WHERE で行を絞り込む

| 演算子 | 意味 | 例 |
|---|---|---|
| `=`, `<>`（または `!=`） | 等しい、等しくない | `category = '飲料'` |
| `<`, `<=`, `>`, `>=` | 大小比較 | `price >= 150` |
| `AND`, `OR`, `NOT` | 条件の組み合わせ | `price >= 100 AND category = '果物'` |
| `IN (...)` | いずれかに一致 | `category IN ('果物', 'パン')` |
| `BETWEEN a AND b` | 範囲（両端を含む） | `price BETWEEN 100 AND 200` |
| `LIKE` | パターン一致（`%` は任意の文字列） | `product_name LIKE '%茶'` |
| `IS NULL` | 欠損値かどうか | `price IS NULL` |

文字列は **シングルクォート** `'...'` で囲みます。

In [ ]:
duckdb.sql("SELECT * FROM products WHERE category = '飲料'").df()

In [ ]:
duckdb.sql("""
SELECT product_name, category, price
FROM products
WHERE price >= 100 AND category <> '飲料'
""").df()

In [ ]:
print(duckdb.sql("SELECT product_name FROM products WHERE category IN ('果物', 'パン')").df())
print(duckdb.sql("SELECT product_name, price FROM products WHERE price BETWEEN 100 AND 200").df())
print(duckdb.sql("SELECT product_name FROM products WHERE product_name LIKE '%茶'").df())

### 3.3 ORDER BY と LIMIT

In [ ]:
# 単価の高い順（DESC）に並べ替え、上位 3 件
duckdb.sql("""
SELECT product_name, price
FROM products
ORDER BY price DESC
LIMIT 3
""").df()

In [ ]:
# 複数の列で並べ替え：カテゴリの昇順 → 同じカテゴリ内では単価の降順
duckdb.sql("""
SELECT category, product_name, price
FROM products
ORDER BY category ASC, price DESC
""").df()

### 3.4 日付の扱い

`sales` の `sale_date` は日付型です。`year()`、`month()`、`strftime()` などの関数で年・月を取り出したり、
文字列 `'2025-03-01'` と比較したりできます。

In [ ]:
duckdb.sql("""
SELECT sale_id, sale_date, year(sale_date) AS y, month(sale_date) AS m, strftime(sale_date, '%Y-%m') AS ym
FROM sales
LIMIT 5
""").df()

In [ ]:
# 2025 年 3 月の売上明細だけを取り出す
march = duckdb.sql("""
SELECT *
FROM sales
WHERE sale_date >= '2025-03-01' AND sale_date < '2025-04-01'
ORDER BY sale_date
""").df()
print("3 月の件数:", len(march))
march.head()

### 練習問題 1

1. `products` から、単価が 150 円未満の商品の名前と単価を、単価の安い順に表示してください。
2. `sales` から、数量（quantity）が 8 以上の売上を、日付の新しい順に 5 件表示してください。
3. `products` から、商品名に「ん」を含む商品を表示してください（`LIKE`）。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```python
# 1
print(duckdb.sql("""
SELECT product_name, price FROM products
WHERE price < 150
ORDER BY price
""").df())

# 2
print(duckdb.sql("""
SELECT * FROM sales
WHERE quantity >= 8
ORDER BY sale_date DESC
LIMIT 5
""").df())

# 3
print(duckdb.sql("SELECT * FROM products WHERE product_name LIKE '%ん%'").df())
```

</details>

---
## 4. 集計（GROUP BY）

**集計関数** は、複数の行を 1 つの値にまとめます。

| 関数 | 意味 |
|---|---|
| `COUNT(*)` | 行数 |
| `SUM(列)` | 合計 |
| `AVG(列)` | 平均 |
| `MIN(列)`, `MAX(列)` | 最小・最大 |

`GROUP BY 列` を付けると、その列の値ごとに集計します（pandas の `groupby` に相当）。

In [ ]:
# 表全体の集計
duckdb.sql("""
SELECT COUNT(*) AS n_sales, SUM(quantity) AS total_qty, AVG(quantity) AS avg_qty,
       MIN(sale_date) AS first_day, MAX(sale_date) AS last_day
FROM sales
""").df()

In [ ]:
# 店舗ごとの売上件数と合計数量
duckdb.sql("""
SELECT store_id, COUNT(*) AS n_sales, SUM(quantity) AS total_qty
FROM sales
GROUP BY store_id
ORDER BY store_id
""").df()

In [ ]:
# 月ごとの合計数量（strftime で年月の文字列を作って GROUP BY）
duckdb.sql("""
SELECT strftime(sale_date, '%Y-%m') AS ym, COUNT(*) AS n_sales, SUM(quantity) AS total_qty
FROM sales
GROUP BY ym
ORDER BY ym
""").df()

### 4.1 HAVING：集計結果で絞り込む

`WHERE` は **集計の前** に行を絞り込み、`HAVING` は **集計の後** にグループを絞り込みます。

In [ ]:
# 合計数量が 350 以上の商品だけ
duckdb.sql("""
SELECT product_id, SUM(quantity) AS total_qty
FROM sales
GROUP BY product_id
HAVING SUM(quantity) >= 350
ORDER BY total_qty DESC
""").df()

In [ ]:
# WHERE と HAVING の組み合わせ：1〜3 月に限定した上で、件数 30 以上の店舗
duckdb.sql("""
SELECT store_id, COUNT(*) AS n_sales
FROM sales
WHERE sale_date < '2025-04-01'
GROUP BY store_id
HAVING COUNT(*) >= 30
ORDER BY n_sales DESC
""").df()

### 4.2 複数の列で GROUP BY

In [ ]:
duckdb.sql("""
SELECT store_id, product_id, SUM(quantity) AS total_qty
FROM sales
GROUP BY store_id, product_id
ORDER BY store_id, product_id
""").df().head(12)

### 練習問題 2

1. `sales` から、商品（product_id）ごとの売上件数と平均数量（小数第 1 位まで、`ROUND(AVG(quantity), 1)`）を求めてください。
2. 店舗ごとの合計数量を求め、合計が 400 以上の店舗だけを表示してください。
3. 月ごとの売上件数を求め、件数の多い順に並べてください。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```python
# 1
print(duckdb.sql("""
SELECT product_id, COUNT(*) AS n_sales, ROUND(AVG(quantity), 1) AS avg_qty
FROM sales GROUP BY product_id ORDER BY product_id
""").df())

# 2
print(duckdb.sql("""
SELECT store_id, SUM(quantity) AS total_qty
FROM sales GROUP BY store_id
HAVING SUM(quantity) >= 400
""").df())

# 3
print(duckdb.sql("""
SELECT strftime(sale_date, '%Y-%m') AS ym, COUNT(*) AS n_sales
FROM sales GROUP BY ym ORDER BY n_sales DESC
""").df())
```

</details>

---
## 5. 表の結合（JOIN）

`sales` には `product_id` と `store_id` しかなく、商品名や店舗名はありません。別の表にある情報を
ID で結び付けて取り出すのが **JOIN** です。

```sql
SELECT ...
FROM   表A
JOIN   表B ON 表A.キー = 表B.キー
```

| 種類 | 意味 |
|---|---|
| `JOIN`（`INNER JOIN`） | 両方の表に存在する行だけ |
| `LEFT JOIN` | 左の表の行はすべて残し、右に対応がなければ NULL |

表に **別名**（`sales AS s` や単に `sales s`）を付けると、`s.quantity` のように短く書けます。

### 5.1 2 つの表を結合する

In [ ]:
duckdb.sql("""
SELECT s.sale_id, s.sale_date, p.product_name, p.price, s.quantity
FROM sales AS s
JOIN products AS p ON s.product_id = p.product_id
ORDER BY s.sale_id
LIMIT 5
""").df()

### 5.2 3 つの表を結合し、売上金額を計算する

売上金額 = 単価 × 数量 です。JOIN した結果に対して集計もできます。

In [ ]:
duckdb.sql("""
SELECT s.sale_id, s.sale_date, st.store_name, p.product_name, p.price * s.quantity AS amount
FROM sales AS s
JOIN products AS p ON s.product_id = p.product_id
JOIN stores AS st ON s.store_id = st.store_id
ORDER BY s.sale_id
LIMIT 5
""").df()

In [ ]:
# カテゴリ別の売上金額
duckdb.sql("""
SELECT p.category, SUM(p.price * s.quantity) AS revenue
FROM sales AS s
JOIN products AS p ON s.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue DESC
""").df()

In [ ]:
# 地域 × カテゴリ別の売上金額
duckdb.sql("""
SELECT st.region, p.category, SUM(p.price * s.quantity) AS revenue
FROM sales AS s
JOIN products AS p ON s.product_id = p.product_id
JOIN stores AS st ON s.store_id = st.store_id
GROUP BY st.region, p.category
ORDER BY st.region, revenue DESC
""").df()

### 5.3 LEFT JOIN：売れていない商品も残す

`products` には一度も売れていない「紅茶」（product_id = 7）があります。`JOIN` では消えてしまいますが、
`LEFT JOIN` なら残ります（売上は NULL → `COALESCE` で 0 に置き換え）。

In [ ]:
duckdb.sql("""
SELECT p.product_name, COUNT(s.sale_id) AS n_sales, COALESCE(SUM(s.quantity), 0) AS total_qty
FROM products AS p
LEFT JOIN sales AS s ON p.product_id = s.product_id
GROUP BY p.product_name
ORDER BY total_qty DESC
""").df()

### 練習問題 3

1. 店舗名ごとの売上金額（単価 × 数量の合計）を、金額の大きい順に表示してください。
2. 商品名ごとの売上金額と販売数量を求め、売上金額の上位 3 商品を表示してください。
3. 地域（region）ごとの売上件数と売上金額を求めてください。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```python
# 1
print(duckdb.sql("""
SELECT st.store_name, SUM(p.price * s.quantity) AS revenue
FROM sales s
JOIN products p ON s.product_id = p.product_id
JOIN stores st ON s.store_id = st.store_id
GROUP BY st.store_name ORDER BY revenue DESC
""").df())

# 2
print(duckdb.sql("""
SELECT p.product_name, SUM(p.price * s.quantity) AS revenue, SUM(s.quantity) AS total_qty
FROM sales s JOIN products p ON s.product_id = p.product_id
GROUP BY p.product_name ORDER BY revenue DESC LIMIT 3
""").df())

# 3
print(duckdb.sql("""
SELECT st.region, COUNT(*) AS n_sales, SUM(p.price * s.quantity) AS revenue
FROM sales s
JOIN products p ON s.product_id = p.product_id
JOIN stores st ON s.store_id = st.store_id
GROUP BY st.region
""").df())
```

</details>

---
## 6. サブクエリと CTE（WITH 句）

クエリの中に別のクエリを入れることを **サブクエリ** といいます。長くなりがちなので、
**CTE（Common Table Expression, `WITH` 句）** で「一時的な表」に名前を付けると読みやすくなります。

### 6.1 WHERE 句の中のサブクエリ

In [ ]:
# 平均単価より高い商品
duckdb.sql("""
SELECT product_name, price
FROM products
WHERE price > (SELECT AVG(price) FROM products)
ORDER BY price DESC
""").df()

In [ ]:
# 愛知県の店舗で売れた売上だけを数える（IN + サブクエリ）
duckdb.sql("""
SELECT COUNT(*) AS n_sales_aichi
FROM sales
WHERE store_id IN (SELECT store_id FROM stores WHERE region = '愛知')
""").df()

### 6.2 FROM 句の中のサブクエリ

In [ ]:
# 店舗ごとの合計数量を求めてから、その平均を取る
duckdb.sql("""
SELECT AVG(total_qty) AS avg_total_qty_per_store
FROM (
    SELECT store_id, SUM(quantity) AS total_qty
    FROM sales
    GROUP BY store_id
) AS t
""").df()

### 6.3 CTE（WITH 句）で読みやすく

同じ処理を `WITH` で書き直します。複数の CTE をカンマで並べることもできます。

In [ ]:
duckdb.sql("""
WITH store_sales AS (
    SELECT s.store_id, st.store_name, SUM(p.price * s.quantity) AS revenue
    FROM sales AS s
    JOIN products AS p ON s.product_id = p.product_id
    JOIN stores AS st ON s.store_id = st.store_id
    GROUP BY s.store_id, st.store_name
),
overall AS (
    SELECT AVG(revenue) AS avg_revenue FROM store_sales
)
SELECT ss.store_name, ss.revenue, ROUND(ss.revenue - o.avg_revenue) AS diff_from_avg
FROM store_sales AS ss, overall AS o
ORDER BY ss.revenue DESC
""").df()

### 練習問題 4

1. 数量（quantity）の平均より多く売れた売上明細の件数を、サブクエリを使って求めてください。
2. `WITH` 句を使って「商品ごとの売上金額」の一時表を作り、そのうち売上金額が 20,000 円以上の商品を表示してください。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```python
# 1
print(duckdb.sql("""
SELECT COUNT(*) AS n
FROM sales
WHERE quantity > (SELECT AVG(quantity) FROM sales)
""").df())

# 2
print(duckdb.sql("""
WITH product_sales AS (
    SELECT p.product_name, SUM(p.price * s.quantity) AS revenue
    FROM sales s JOIN products p ON s.product_id = p.product_id
    GROUP BY p.product_name
)
SELECT * FROM product_sales WHERE revenue >= 20000 ORDER BY revenue DESC
""").df())
```

</details>

---
## 7. ウィンドウ関数（累積和・移動平均・ランキング）

**ウィンドウ関数** は、「行をまとめずに」各行に集計値を付け加える機能です。経済の時系列データでよく使う
累積和、移動平均、前期比、順位付けが 1 つの SQL で書けます。

```sql
関数() OVER (PARTITION BY グループ列 ORDER BY 並べ替え列 [ROWS BETWEEN ...])
```

まず、架空の月次 GDP（単位：兆円）の時系列を作ります。

In [ ]:
months = pd.date_range("2023-01-01", periods=36, freq="MS")   # 月初の日付 36 か月分
trend = 500 + np.arange(36) * 1.2                              # 緩やかな成長トレンド
noise = rng.normal(0, 4, 36)
gdp = pd.DataFrame({"month_start": months, "gdp": (trend + noise).round(1)})
gdp.head()

### 7.1 累積和と移動平均

In [ ]:
duckdb.sql("""
SELECT month_start, gdp,
       SUM(gdp) OVER (ORDER BY month_start) AS cum_gdp,
       ROUND(AVG(gdp) OVER (ORDER BY month_start ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS ma3
FROM gdp
ORDER BY month_start
""").df().head(8)

- `SUM(...) OVER (ORDER BY month_start)`：先頭からその行までの累積和
- `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW`：「2 行前から現在の行まで」= 直近 3 か月の移動平均

### 7.2 前期比（LAG）

`LAG(列)` は 1 行前の値を返します（`LEAD` は 1 行後）。前期比成長率が計算できます。

In [ ]:
growth = duckdb.sql("""
SELECT month_start, gdp,
       LAG(gdp) OVER (ORDER BY month_start) AS prev_gdp,
       ROUND(100.0 * (gdp - LAG(gdp) OVER (ORDER BY month_start)) / LAG(gdp) OVER (ORDER BY month_start), 2) AS growth_pct
FROM gdp
ORDER BY month_start
""").df()
growth.head(6)

In [ ]:
# 前年同月比：12 行前の値と比較する
duckdb.sql("""
SELECT month_start, gdp,
       LAG(gdp, 12) OVER (ORDER BY month_start) AS gdp_last_year,
       ROUND(100.0 * (gdp / LAG(gdp, 12) OVER (ORDER BY month_start) - 1), 2) AS yoy_pct
FROM gdp
ORDER BY month_start
""").df().tail(5)

### 7.3 ランキング（RANK / ROW_NUMBER）と PARTITION BY

`PARTITION BY` を付けると、グループごとに独立して順位を付けられます。

In [ ]:
duckdb.sql("""
WITH store_sales AS (
    SELECT st.region, st.store_name, SUM(p.price * s.quantity) AS revenue
    FROM sales AS s
    JOIN products AS p ON s.product_id = p.product_id
    JOIN stores AS st ON s.store_id = st.store_id
    GROUP BY st.region, st.store_name
)
SELECT region, store_name, revenue,
       RANK() OVER (ORDER BY revenue DESC) AS overall_rank,
       RANK() OVER (PARTITION BY region ORDER BY revenue DESC) AS rank_in_region
FROM store_sales
ORDER BY overall_rank
""").df()

In [ ]:
# 店舗ごとに「売上金額が最も大きかった商品」を 1 つ選ぶ（ROW_NUMBER で 1 位だけ残す）
duckdb.sql("""
WITH sp AS (
    SELECT st.store_name, p.product_name, SUM(p.price * s.quantity) AS revenue,
           ROW_NUMBER() OVER (PARTITION BY st.store_name ORDER BY SUM(p.price * s.quantity) DESC) AS rn
    FROM sales AS s
    JOIN products AS p ON s.product_id = p.product_id
    JOIN stores AS st ON s.store_id = st.store_id
    GROUP BY st.store_name, p.product_name
)
SELECT store_name, product_name, revenue
FROM sp
WHERE rn = 1
ORDER BY store_name
""").df()

### 練習問題 5

1. `gdp` に対して、6 か月移動平均（`ROWS BETWEEN 5 PRECEDING AND CURRENT ROW`）の列を追加してください。
2. 月ごとの売上金額（sales × products）を求め、その累積和の列を追加してください。
3. 商品ごとの売上金額に `RANK()` で順位を付けてください。

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```python
# 1
print(duckdb.sql("""
SELECT month_start, gdp,
       ROUND(AVG(gdp) OVER (ORDER BY month_start ROWS BETWEEN 5 PRECEDING AND CURRENT ROW), 1) AS ma6
FROM gdp ORDER BY month_start
""").df().head(8))

# 2
print(duckdb.sql("""
WITH monthly AS (
    SELECT strftime(s.sale_date, '%Y-%m') AS ym, SUM(p.price * s.quantity) AS revenue
    FROM sales s JOIN products p ON s.product_id = p.product_id
    GROUP BY ym
)
SELECT ym, revenue, SUM(revenue) OVER (ORDER BY ym) AS cum_revenue
FROM monthly ORDER BY ym
""").df())

# 3
print(duckdb.sql("""
SELECT p.product_name, SUM(p.price * s.quantity) AS revenue,
       RANK() OVER (ORDER BY SUM(p.price * s.quantity) DESC) AS rnk
FROM sales s JOIN products p ON s.product_id = p.product_id
GROUP BY p.product_name ORDER BY rnk
""").df())
```

</details>

---
## 8. CSV ファイルに直接クエリする

DuckDB は、CSV ファイルを **読み込む手順なしで** そのまま表として扱えます。ファイル名を `FROM '...'` に書くだけです。
まず、`sales` を CSV に保存します（JupyterLite では、左のファイルブラウザにファイルが現れます）。

In [ ]:
sales.to_csv("sales.csv", index=False)
products.to_csv("products.csv", index=False)
print("CSV を書き出しました")

In [ ]:
# ファイル名をそのまま表として使う（列の型は自動で推定される）
duckdb.sql("SELECT * FROM 'sales.csv' LIMIT 5").df()

In [ ]:
# CSV 同士を JOIN して集計することもできる
duckdb.sql("""
SELECT p.category, SUM(p.price * s.quantity) AS revenue
FROM 'sales.csv' AS s
JOIN 'products.csv' AS p ON s.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue DESC
""").df()

In [ ]:
# 列の型を確認する（DESCRIBE）
duckdb.sql("DESCRIBE SELECT * FROM 'sales.csv'").df()

結果をファイルに保存したいときは、いったん `.df()` で DataFrame にしてから `to_csv()` を使うのが簡単です。
（DuckDB 自体にも `COPY (SELECT ...) TO 'out.csv' (HEADER)` という構文があります。）

In [ ]:
category_revenue = duckdb.sql("""
SELECT p.category, SUM(p.price * s.quantity) AS revenue
FROM 'sales.csv' AS s JOIN 'products.csv' AS p ON s.product_id = p.product_id
GROUP BY p.category ORDER BY revenue DESC
""").df()
category_revenue.to_csv("category_revenue.csv", index=False)
print(open("category_revenue.csv", encoding="utf-8").read())

---
## 9. インメモリ・データベース（CREATE / INSERT / UPDATE / DELETE）

ここまでは「DataFrame に SQL を投げる」使い方でしたが、DuckDB は本物のデータベースとしても使えます。
`duckdb.connect()` でメモリ上にデータベースを作り、表を作成・更新してみましょう。

| 文 | 役割 |
|---|---|
| `CREATE TABLE` | 表を作る |
| `INSERT INTO` | 行を追加する |
| `UPDATE ... SET ... WHERE` | 行を更新する |
| `DELETE FROM ... WHERE` | 行を削除する |

`con.execute(SQL)` で文を実行し、`con.sql(SQL).df()` で結果を DataFrame として取り出します。

In [ ]:
con = duckdb.connect()   # 引数なし = メモリ上のデータベース（ファイル名を渡すとファイルに保存される）

# 列の型を指定して表を作る
con.execute("""
CREATE TABLE product_master (
    product_id INTEGER PRIMARY KEY,
    product_name VARCHAR,
    category VARCHAR,
    price INTEGER
)
""")
print(con.sql("SELECT * FROM product_master").df())   # まだ空

In [ ]:
# 行を追加する
con.execute("INSERT INTO product_master VALUES (1, 'りんご', '果物', 128)")
con.execute("INSERT INTO product_master VALUES (2, 'バナナ', '果物', 98), (3, 'みかん', '果物', 60)")
con.sql("SELECT * FROM product_master").df()

In [ ]:
# DataFrame から一気に表を作ることもできる（CREATE TABLE ... AS SELECT）
con.execute("CREATE TABLE store_master AS SELECT * FROM stores")
con.execute("CREATE TABLE sales_log AS SELECT * FROM sales")
print(con.sql("SELECT COUNT(*) AS n FROM sales_log").df())
con.sql("SELECT * FROM store_master").df()

In [ ]:
# 更新と削除
con.execute("UPDATE product_master SET price = 138 WHERE product_id = 1")
con.execute("DELETE FROM product_master WHERE product_id = 3")
con.sql("SELECT * FROM product_master").df()

In [ ]:
# パラメータ付きのクエリ（? に値を差し込む。文字列を直接つなげるより安全）
new_products = [(4, "牛乳", "飲料", 210), (5, "パン", "パン", 180)]
for row in new_products:
    con.execute("INSERT INTO product_master VALUES (?, ?, ?, ?)", row)

threshold = 150
con.execute("SELECT product_name, price FROM product_master WHERE price >= ?", [threshold]).df()

In [ ]:
# 表の一覧を確認し、最後に接続を閉じる
print(con.sql("SHOW TABLES").df())
con.close()

### 練習問題 6

1. 新しいメモリ DB を作り、`students(student_id INTEGER, name VARCHAR, score INTEGER)` という表を作成して 4 人分のデータを追加してください。
2. 点数が 60 未満の学生の点数を 60 に更新（UPDATE）し、全員の平均点を求めてください。
3. 表を DataFrame として取り出し、CSV に保存してください。

In [ ]:
# 練習問題 6 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 6 の解答例を見る</strong></summary>

```python
# 1
c = duckdb.connect()
c.execute("CREATE TABLE students (student_id INTEGER, name VARCHAR, score INTEGER)")
c.execute("INSERT INTO students VALUES (1, '田中', 78), (2, '鈴木', 55), (3, '佐藤', 92), (4, '高橋', 48)")
print(c.sql("SELECT * FROM students").df())

# 2
c.execute("UPDATE students SET score = 60 WHERE score < 60")
print(c.sql("SELECT AVG(score) AS avg_score FROM students").df())

# 3
c.sql("SELECT * FROM students").df().to_csv("students.csv", index=False)
c.close()
```

</details>

---
## 10. pandas との往復とグラフ化

SQL で集計した結果は `.df()` で DataFrame になるので、pandas の機能や matplotlib でそのまま加工・可視化できます。
「SQL で集計 → pandas で整形 → matplotlib で描画」は分析の定番の流れです。

In [ ]:
monthly = duckdb.sql("""
SELECT strftime(s.sale_date, '%Y-%m') AS ym, st.region, SUM(p.price * s.quantity) AS revenue
FROM sales AS s
JOIN products AS p ON s.product_id = p.product_id
JOIN stores AS st ON s.store_id = st.store_id
GROUP BY ym, st.region
ORDER BY ym, st.region
""").df()

# pandas で「行 = 月、列 = 地域」の形に並べ替える
pivot = monthly.pivot(index="ym", columns="region", values="revenue")
print(pivot)

In [ ]:
pivot.plot(kind="line", marker="o", figsize=(8, 4))
plt.title("地域別・月別売上金額")
plt.xlabel("年月")
plt.ylabel("売上金額（円）")
plt.grid(True)
plt.show()

In [ ]:
category_revenue.plot(kind="bar", x="category", y="revenue", legend=False, figsize=(6, 4), color="steelblue")
plt.title("カテゴリ別売上金額")
plt.xlabel("カテゴリ")
plt.ylabel("売上金額（円）")
plt.show()

逆に、pandas で加工した DataFrame は、そのまま次の SQL の表として使えます。

In [ ]:
# pandas で「単価 150 円以上の商品」だけの DataFrame を作り、それを SQL で集計する
expensive = products[products["price"] >= 150]

duckdb.sql("""
SELECT e.product_name, SUM(s.quantity) AS total_qty
FROM sales AS s
JOIN expensive AS e ON s.product_id = e.product_id
GROUP BY e.product_name
ORDER BY total_qty DESC
""").df()

---
## 11. 標準ライブラリ sqlite3 との比較

Python には最初から **SQLite** というデータベースが付属しています（`sqlite3` モジュール）。
SQL の文法はほぼ同じですが、使い方が少し違います。

| 項目 | DuckDB | sqlite3 |
|---|---|---|
| インストール | `import duckdb`（JupyterLite に同梱） | 標準ライブラリ（何もしなくても使える） |
| DataFrame への SQL | `duckdb.sql("... FROM df")` で直接 | `df.to_sql()` で表に入れてから |
| 結果を DataFrame に | `.df()` | `pd.read_sql(SQL, 接続)` |
| 得意分野 | 大量データの集計・分析 | 小さなアプリのデータ保存 |
| 日付関数 | `strftime(date, '%Y-%m')`, `year(date)` | `strftime('%Y-%m', date)`（引数の順番が逆） |

同じ集計を sqlite3 でやってみましょう。

In [ ]:
import sqlite3

sq = sqlite3.connect(":memory:")               # メモリ上の SQLite DB
sales.to_sql("sales", sq, index=False)          # DataFrame を表として書き込む
products.to_sql("products", sq, index=False)

query = """
SELECT p.category, SUM(p.price * s.quantity) AS revenue
FROM sales AS s
JOIN products AS p ON s.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue DESC
"""
pd.read_sql(query, sq)

In [ ]:
# 日付関数の書き方が違う点に注意（SQLite は strftime(書式, 日付)）
pd.read_sql("""
SELECT strftime('%Y-%m', sale_date) AS ym, COUNT(*) AS n_sales
FROM sales
GROUP BY ym
ORDER BY ym
""", sq)

In [ ]:
sq.close()
print("同じ SQL が DuckDB でも動くことを確認:")
duckdb.sql(query).df()

---
## 12. SQL と pandas の対応表

| やりたいこと | SQL | pandas |
|---|---|---|
| 列を選ぶ | `SELECT a, b FROM df` | `df[["a", "b"]]` |
| 行を絞る | `WHERE a > 10` | `df[df["a"] > 10]` |
| 並べ替え | `ORDER BY a DESC` | `df.sort_values("a", ascending=False)` |
| 先頭 n 件 | `LIMIT 5` | `df.head(5)` |
| 重複を除く | `SELECT DISTINCT a` | `df["a"].unique()` |
| 集計 | `SELECT g, SUM(a) FROM df GROUP BY g` | `df.groupby("g")["a"].sum()` |
| 集計後の絞り込み | `HAVING SUM(a) > 100` | `s = df.groupby("g")["a"].sum(); s[s > 100]` |
| 結合 | `JOIN t ON df.k = t.k` | `df.merge(t, on="k")` |
| 左結合 | `LEFT JOIN` | `df.merge(t, on="k", how="left")` |
| 新しい列 | `SELECT a * 2 AS b` | `df["b"] = df["a"] * 2` |
| 累積和 | `SUM(a) OVER (ORDER BY t)` | `df["a"].cumsum()` |
| 移動平均 | `AVG(a) OVER (... ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` | `df["a"].rolling(3).mean()` |
| 前期の値 | `LAG(a) OVER (ORDER BY t)` | `df["a"].shift(1)` |
| 順位 | `RANK() OVER (ORDER BY a DESC)` | `df["a"].rank(ascending=False)` |

どちらでもできますが、**複数の表を組み合わせた集計は SQL、細かい加工や可視化は pandas** が得意です。

---
## 13. まとめ

| トピック | 主な構文 |
|---|---|
| 基本 | `SELECT ... FROM ... WHERE ... ORDER BY ... LIMIT` |
| 集計 | `COUNT / SUM / AVG / MIN / MAX`, `GROUP BY`, `HAVING` |
| 結合 | `JOIN ... ON`, `LEFT JOIN`, `COALESCE` |
| サブクエリ | `WHERE a > (SELECT ...)`, `FROM (SELECT ...) AS t`, `WITH name AS (...)` |
| ウィンドウ関数 | `SUM() OVER (ORDER BY ...)`, `AVG() OVER (... ROWS BETWEEN ...)`, `LAG()`, `RANK() OVER (PARTITION BY ...)` |
| ファイル | `FROM 'file.csv'`, `DESCRIBE` |
| データベース操作 | `duckdb.connect()`, `CREATE TABLE`, `INSERT`, `UPDATE`, `DELETE`, `con.sql().df()` |
| pandas 連携 | `duckdb.sql("... FROM df").df()`, `pd.read_sql()`（sqlite3） |

## 次のステップ

- `python/pandas/pandas_intermediate_tutorial.ipynb` — groupby・結合・時系列を pandas 側から
- `python/polars/`（今後追加予定）— DuckDB と相性の良い高速 DataFrame ライブラリ
- `python/altair/altair_beginner_tutorial.ipynb` — SQL で集計した結果を対話的なグラフにする

---
## 総合演習：店舗別・商品別の売上分析レポート

これまで学んだ SQL を組み合わせて、次の分析を行ってください（表は `sales`, `products`, `stores` を使います）。

1. `WITH` 句で「売上明細に商品名・店舗名・地域・売上金額（単価 × 数量）を付けた表」`detail` を作り、先頭 5 行を表示する。
2. 地域ごと・月ごとの売上金額を求め、地域ごとに **月別売上の累積和** の列を付ける。
3. 商品ごとの売上金額に順位を付け、上位 3 商品を表示する。
4. 店舗ごとの売上金額と、全店舗平均との差を表示する。
5. 3 の結果（商品別売上ランキング）を `product_ranking.csv` に保存し、棒グラフにする。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください

### 総合演習の解答例

自分で書いてから、次のセルを実行して結果を比べてみてください。

In [ ]:
# 1. 明細表を作る
detail_sql = """
SELECT s.sale_id, s.sale_date, strftime(s.sale_date, '%Y-%m') AS ym,
       st.store_name, st.region, p.product_name, p.category, p.price * s.quantity AS amount
FROM sales AS s
JOIN products AS p ON s.product_id = p.product_id
JOIN stores AS st ON s.store_id = st.store_id
"""
print(duckdb.sql(detail_sql + " ORDER BY s.sale_id LIMIT 5").df())

# 2. 地域 × 月の売上と累積和
print(duckdb.sql(f"""
WITH detail AS ({detail_sql}),
monthly AS (
    SELECT region, ym, SUM(amount) AS revenue
    FROM detail GROUP BY region, ym
)
SELECT region, ym, revenue,
       SUM(revenue) OVER (PARTITION BY region ORDER BY ym) AS cum_revenue
FROM monthly ORDER BY region, ym
""").df())

# 3. 商品ランキング
ranking = duckdb.sql(f"""
WITH detail AS ({detail_sql})
SELECT product_name, SUM(amount) AS revenue,
       RANK() OVER (ORDER BY SUM(amount) DESC) AS rnk
FROM detail GROUP BY product_name ORDER BY rnk
""").df()
print(ranking.head(3))

# 4. 店舗別売上と平均との差
print(duckdb.sql(f"""
WITH detail AS ({detail_sql}),
store_rev AS (SELECT store_name, SUM(amount) AS revenue FROM detail GROUP BY store_name)
SELECT store_name, revenue, ROUND(revenue - AVG(revenue) OVER ()) AS diff_from_avg
FROM store_rev ORDER BY revenue DESC
""").df())

# 5. CSV に保存して棒グラフ
ranking.to_csv("product_ranking.csv", index=False)
ranking.plot(kind="bar", x="product_name", y="revenue", legend=False, figsize=(7, 4), color="steelblue")
plt.title("商品別売上金額ランキング")
plt.xlabel("商品")
plt.ylabel("売上金額（円）")
plt.show()

お疲れさまでした！ SQL の基本（SELECT・集計・JOIN・サブクエリ・ウィンドウ関数）と、DuckDB で pandas と
行き来する方法を学びました。実際の分析では、まず SQL で必要なデータを集計し、pandas と matplotlib で
仕上げる流れを意識してみてください。